In [1]:
import tabula
from img2pdf import convert
import os
import pytesseract
from PIL import Image

In [4]:
file_path_test = "C:/pessoal/projetos_individuais/ia/Iso_olhos/dados/dados_pentacam_identificados/1300_Mininel_Ermelinda Aparecida Jacomini_OD_29082024_150150_4 Maps Refr.JPG"
# pytesseract.pytesseract.tesseract_cmd = r'C:/Program Files/Tesseract-OCR/tesseract.exe'


In [ ]:
def load_image(image_path):
    """Load an image from the specified path using Pillow, convert to RGB and returns the image."""
    try:
        image = Image.open(image_path).convert("RGB")
        return image
    except Exception as e:
        print(f"Error loading image: {e}")
        return None


def crop_image(image, crop_box):
    """Crop the image using the specified crop box (left, upper, right, lower)."""
    try:
        cropped_image = image.crop(crop_box)
        return cropped_image
    except Exception as e:
        print(f"Error cropping image: {e}")
        return None

In [ ]:
"""
extract_pentacam.py

Extrai os valores textuais das tabelas à esquerda de um relatório Pentacam (PNG/JPG).
Saída: JSON e CSV com os pares "campo": "valor".
Gera imagens de debug (crop, preprocess, boxes).
"""

import os
import re
import json
import argparse
from collections import defaultdict
from PIL import Image, ImageDraw
import numpy as np
import cv2
import pytesseract
import pandas as pd

# === CONFIG ===
# Se estiver no Windows e o tesseract não estiver no PATH, defina o caminho completo:
TESSERACT_CMD = None  # exemplo: r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# Crop da região esquerda como fração da largura (ajuste se necessário)
LEFT_CROP_FRAC = 0.32
# Tesseract config extras
TESSERACT_LANG = 'por+eng'  # use 'eng' se não tiver português
TESSERACT_PSM = 6  # 6 = assume um bloco uniforme de texto
# =================

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

def load_image(path):
    img = Image.open(path).convert('RGB')
    return img

def crop_left_region(img, frac=LEFT_CROP_FRAC):
    w,h = img.size
    box = (0, 0, int(w*frac), h)
    return img.crop(box)

def preprocess_for_ocr(pil_img):
    # Converte para OpenCV
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # Aumentar contraste via CLAHE
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    # Suavizar levemente e aplicar sharpen opcional
    gray = cv2.medianBlur(gray, 3)
    # Opcional: binarização adaptativa (comentar/descomentar conforme a imagem)
    # gray = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    #                              cv2.THRESH_BINARY, 15, 3)
    return gray

def ocr_with_data(gray_img):
    # converte de volta para PIL para pytesseract, mas pode aceitar numpy array diretamente
    pil = Image.fromarray(gray_img)
    custom_config = f'--oem 3 --psm {TESSERACT_PSM}'
    data = pytesseract.image_to_data(pil, lang=TESSERACT_LANG, config=custom_config, output_type=pytesseract.Output.DICT)
    return data

def reconstruct_lines(ocr_data, line_bucket=8):
    # Agrupa as caixas por coordenada y (aproximação de linhas)
    lines = defaultdict(list)
    n = len(ocr_data['level'])
    for i in range(n):
        txt = ocr_data['text'][i].strip()
        if not txt:
            continue
        top = ocr_data['top'][i]
        left = ocr_data['left'][i]
        key = int(top / line_bucket)
        lines[key].append((left, txt, i))
    ordered_lines = []
    for k in sorted(lines.keys()):
        parts = sorted(lines[k], key=lambda x: x[0])
        line_text = " ".join([p[1] for p in parts])
        indices = [p[2] for p in parts]
        # compute average conf
        confs = [int(ocr_data['conf'][idx]) if ocr_data['conf'][idx].isdigit() else -1 for idx in indices]
        avg_conf = sum([c for c in confs if c>=0])/len([c for c in confs if c>=0]) if any(c>=0 for c in confs) else None
        ordered_lines.append({'text': line_text, 'conf': avg_conf, 'indices': indices})
    return ordered_lines

# Dicionário de regex para campos comuns. Expandir conforme necessário.
FIELD_PATTERNS = {
    'K1': r'(?:K\s*1|K1|K 1)[:\s]*([0-9]{1,2}\.?[0-9]{0,2})\s*D?',
    'K2': r'(?:K\s*2|K2|K 2)[:\s]*([0-9]{1,2}\.?[0-9]{0,2})\s*D?',
    'Km': r'(?:Km|Km[:\s]*)([0-9]{1,2}\.?[0-9]{0,2})\s*D?',
    'Kmax': r'(?:K\s*Max|K\s*Máx|KMax|K\.Max|K Máx)[:\s]*([0-9]{1,2}\.?[0-9]{0,2})',
    'Rp': r'(?:Rp[:\s]*)([0-9]{1,2}\.?[0-9]{0,2})\s*mm?',
    'Rc': r'(?:Rc[:\s]*)([0-9]{1,2}\.?[0-9]{0,2})\s*mm?',
    'Rm': r'(?:Rm[:\s]*)([0-9]{1,2}\.?[0-9]{0,2})\s*mm?',
    'Astig': r'(?:Astig|Astig\.)[:\s]*([0-9]{1,2}\.?[0-9]{0,2})\s*D?',
    'Eixo': r'(?:Eixo|Eixo\s*\(?plano\)?)[:\s]*([0-9]{1,3})',
    'CentroPupilar': r'(?:Centro Pupilar|CentroPupilar)[:\s]*([0-9]{1,4})\s*(?:um|µm|u?m)?',
    'PachyVertex': r'(?:Pachy Vertex N\.?|Pachy Vertex)[:\s]*([0-9]{2,4})\s*(?:um|µm|u?m)?',
    'Ponto_fino': r'(?:Ponto \+ fino|Ponto \+|Ponto\+ fino)[:\s]*([0-9]{2,4})\s*(?:um|µm|u?m)?',
    'Volume_cornea': r'(?:Volume córnea|Volume cornea|Volume Córnea)[:\s]*([0-9]{1,4})',
    'Diam_Pupil': r'(?:Diam\.? Pupill\.?|Diâm\. Pupill\.|Diâm Pupill\.|Diâm Pupilar|Diâmetro Pupilar|Diâm Pupila)[:\s]*([0-9]{1,3}\.?\d?)',
    # fallback to any numeric with units
}

def parse_lines_for_fields(lines):
    found = {}
    # first pass: try pattern matches line-by-line
    for ln in lines:
        text = ln['text']
        for field, pat in FIELD_PATTERNS.items():
            if field in found:
                continue
            m = re.search(pat, text, flags=re.IGNORECASE)
            if m:
                val = m.group(1)
                found[field] = {'value': val, 'line': text, 'conf': ln.get('conf')}
    # second pass: keywords + nearest number fallback
    keywords = {
        'K1': ['k1', 'k 1'],
        'K2': ['k2', 'k 2'],
        'Km': ['km', 'km:'],
        'Rp': ['rp'],
        'Rc': ['rc'],
        'Rm': ['rm'],
        'Astig': ['astig', 'astig.'],
        'Eixo': ['eixo'],
        'CentroPupilar': ['centro pupilar', 'centro_pupilar', 'centro'],
        'PachyVertex': ['pachy', 'pachy vertex'],
        'Ponto_fino': ['ponto', 'ponto +'],
    }
    # extract any numbers with units from line
    for ln in lines:
        txt = ln['text'].lower()
        nums = re.findall(r'[-+]?\d{1,3}\.\d{1,3}|\d{1,4}', txt)
        if not nums:
            continue
        for k, keys in keywords.items():
            if k in found:
                continue
            for kw in keys:
                if kw in txt:
                    # choose the most plausible number (first numeric token)
                    found[k] = {'value': nums[0], 'line': ln['text'], 'conf': ln.get('conf')}
                    break
            if k in found:
                break

    # last resort: scan for explicit "K1 48.1" anywhere in the whole text block
    full_text = "\n".join([ln['text'] for ln in lines])
    for field, pat in FIELD_PATTERNS.items():
        if field in found:
            continue
        m = re.search(pat, full_text, flags=re.IGNORECASE)
        if m:
            found[field] = {'value': m.group(1), 'line': m.group(0), 'conf': None}

    return found

def save_debug_images(original_crop_pil, preprocessed_cv2, ocr_data, outdir):
    os.makedirs(outdir, exist_ok=True)
    left_path = os.path.join(outdir, 'left_crop.png')
    pre_path = os.path.join(outdir, 'left_preprocessed.png')
    boxed_path = os.path.join(outdir, 'left_boxes.png')
    original_crop_pil.save(left_path)
    cv2.imwrite(pre_path, preprocessed_cv2)
    # draw boxes
    img = original_crop_pil.copy()
    draw = ImageDraw.Draw(img)
    n = len(ocr_data['level'])
    for i in range(n):
        txt = ocr_data['text'][i].strip()
        if not txt:
            continue
        x = ocr_data['left'][i]
        y = ocr_data['top'][i]
        w = ocr_data['width'][i]
        h = ocr_data['height'][i]
        conf = int(ocr_data['conf'][i]) if ocr_data['conf'][i].isdigit() else -1
        color = "red" if conf >= 40 else "orange"
        draw.rectangle([x, y, x+w, y+h], outline=color, width=1)
    img.save(boxed_path)
    return left_path, pre_path, boxed_path

def main(args):
    img = load_image(args.image)
    left = crop_left_region(img, frac=args.left_frac)
    pre = preprocess_for_ocr(left)
    ocr_data = ocr_with_data(pre)
    lines = reconstruct_lines(ocr_data)
    found = parse_lines_for_fields(lines)
    # prepare output
    out = {'source_image': args.image, 'fields': found}
    os.makedirs(args.outdir, exist_ok=True)
    json_path = os.path.join(args.outdir, 'pentacam_left_extracted.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, indent=2, ensure_ascii=False)
    # also CSV (field, value, line, conf)
    rows = []
    for k,v in found.items():
        rows.append({'field': k, 'value': v.get('value'), 'line': v.get('line'), 'conf': v.get('conf')})
    df = pd.DataFrame(rows)
    csv_path = os.path.join(args.outdir, 'pentacam_left_extracted.csv')
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')

    # debug images
    left_path, pre_path, boxed_path = save_debug_images(left, pre, ocr_data, args.outdir)

    print("Extração concluída.")
    print("JSON:", json_path)
    print("CSV:", csv_path)
    print("Imagens debug:", left_path, pre_path, boxed_path)
    if args.print_result:
        print("\n=== RESULTADOS ===")
        print(json.dumps(found, indent=2, ensure_ascii=False))

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Extrai campos textuais do lado esquerdo do relatório Pentacam.")
    parser.add_argument('--image', required=True, help='Caminho para a imagem (PNG/JPG).')
    parser.add_argument('--outdir', default='./out_pentacam', help='Diretório de saída.')
    parser.add_argument('--left_frac', type=float, default=LEFT_CROP_FRAC, help='Fraçao da largura a cortar (esquerda).')
    parser.add_argument('--print_result', action='store_true', help='Imprime resultado no console.')
    args = parser.parse_args()
    main(args)

usage: ipykernel_launcher.py [-h] --image IMAGE [--outdir OUTDIR]
                             [--left_frac LEFT_FRAC] [--print_result]
ipykernel_launcher.py: error: the following arguments are required: --image


SystemExit: 2

c:\pessoal\projetos_individuais\ia\Iso_olhos\env\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
